## Assign pca notes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import time
import tensorflow as tf
import keras

from sklearn.model_selection import train_test_split



In [ ]:
cover = pd.read_csv("../../data/covtype.csv")
cover = cover.sample(n=10000, random_state=42)
cover["Cover_Type"] = cover["Cover_Type"]


Inspect the data

In [ ]:
cover.dtypes.value_counts()

In [ ]:
cover.dtypes

In [ ]:
int_cols = cover.select_dtypes(include=['int64'])
val_counts = [cover[c].nunique() for c in cover.columns ]
list(zip(int_cols, val_counts))

In [ ]:
cover["Cover_Type"].unique()

Keras and sklearn classifiers expect labels 0 to 6, or, coded as one hot. We do the former. We can simply subtract one, or use label encoder from  sklearn.preprocessing. If your imported as categorical you will need label encoder

In [ ]:

train_df, test_df = train_test_split(cover, test_size=0.2, random_state=42, stratify=cover["Cover_Type"])
X_train = train_df.drop(columns=["Cover_Type"])
y_train = train_df["Cover_Type"] - 1
X_test = test_df.drop(columns=["Cover_Type"])
y_test = test_df["Cover_Type"] - 1

Or we could have used 

```
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
cover['Cover_Type'] = le.fit_transform(cover['Cover_Type])
```



In [ ]:
from sklearn.preprocessing import StandardScaler

X_train.head()
X_train.iloc[:, :10].head()
scaler = StandardScaler()
X_train.iloc[:, :10] = scaler.fit_transform(X_train.iloc[:, :10])
X_test.iloc[:, :10] = scaler.transform(X_test.iloc[:, :10])

In [ ]:
input_dim = X_train.shape[1]

In [ ]:
enc = keras.models.Sequential(
    [
        keras.layers.InputLayer(shape=(input_dim, )),
        keras.layers.Dense(10, activation= None)
    ]
)

dec = keras.models.Sequential(
    [
        keras.layers.InputLayer(shape=(10, )),
        keras.layers.Dense(7, activation= 'softmax')
    ]
)


mod = keras.models.Sequential(
    [
        enc,
        dec
    ]
)

mod.compile(
    loss=  'sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
mod.fit(
    X_train,
    y_train,
    epochs=  3,
    validation_data=(X_test, y_test)
)

Two ways to do the f1 score- either use sklearn, or one - hot the labels and use categorical crossentropy loss. We show the latter. See the keras docs on this. <https://keras.io/api/losses/probabilistic_losses/#categoricalcrossentropy-class>

But forget about the f1 for this assigment

In [ ]:
y_train = keras.utils.to_categorical(y_train, num_classes=7)
y_test = keras.utils.to_categorical(y_test, num_classes=7)


mod.compile(
    loss='categorical_crossentropy',
    metrics=['accuracy', 
             keras.metrics.F1Score(average='macro', name='f1_macro'),
             keras.metrics.F1Score(average='weighted', name='f1_weighted')]
)


Now do non-linear

In [ ]:
enc = keras.models.Sequential(
    [
        keras.layers.InputLayer(shape=(input_dim, )),
        keras.layers.Dense(10, activation= 'relu')
    ]
)

dec = keras.models.Sequential(
    [
        keras.layers.InputLayer(shape=(10, )),
        keras.layers.Dense(7, activation= 'softmax')
    ]
)


mod = keras.models.Sequential(
    [
        enc,
        dec
    ]
)

mod.compile(
    loss=  'sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
mod.fit(
    X_train,
    y_train,
    epochs=  3,
    validation_data=(X_test, y_test)
)

In [ ]:
enc = keras.models.Sequential(
    [
        keras.layers.InputLayer(shape=(input_dim, )),
        keras.layers.Dense(64, activation= 'relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(32, activation= 'relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(16, activation= 'relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(10, activation= 'relu')
    ]
)

dec = keras.models.Sequential(
    [
        keras.layers.InputLayer(shape=(10, )),
        keras.layers.Dense(7, activation= 'softmax')
    ]
)


mod = keras.models.Sequential(
    [
        enc,
        dec
    ]
)

mod.compile(
    loss=  'sparse_categorical_crossentropy',
    metrics=['accuracy']
)

Really what we want to experiment with is how well we can reconstruct the original data- not a classification. For example the following. Note that we can import layers to save typing

In [ ]:
from keras.layers import InputLayer, Dense, BatchNormalization

enc = keras.models.Sequential(
    [
        InputLayer(shape=(input_dim, )),
        Dense(64, activation= 'relu'),
        BatchNormalization(),
        Dense(32, activation= 'relu'),
        BatchNormalization(),
        Dense(16, activation= 'relu'),
        BatchNormalization(),
        Dense(10, activation= 'relu')
    ]
)

dec = keras.models.Sequential(
    [
        InputLayer(shape=(10, )),
        Dense(16, activation= 'relu'),
        BatchNormalization(),
        Dense(32, activation = 'relu'),
        BatchNormalization(),
        Dense(64, activation = 'relu'),
        BatchNormalization(),
        Dense(input_dim, activation = None)
    ]
)


mod = keras.models.Sequential(
    [
        enc,
        dec
    ]
)

mod.compile(
    loss='mse',
    metrics=['mse']
)

In [ ]:
mod.fit(
    X_train,
    X_train,
    epochs=  3,
    validation_data=(X_test, X_test)
)

Its hard to assess raw mse and loss - it's actually finding mse across all the predictors, for each observation. With image data, however, we can visually compare the reconstructed with the original to get an idea. That is what you do in the Autoencode assignment